# Injection-recovery test
### PS7 Exoplanet Vetting Pipeline

The single most credible evidence of pipeline quality: inject synthetic `batman` transits into a **real quiet light curve** across a grid of period/depth, run the blind detection path, and measure the recovery fraction vs SNR (the completeness curve).

In [ ]:
# --- Setup -------------------------------------------------------------------------
# LOCAL: this notebook imports the `exopipeline` package from the project root.
# COLAB: upload the `exopipeline/` folder (or a zip of it) next to this notebook, then
#        run:  !pip install -q "numpy<2" "astropy<7" lightkurve transitleastsquares \
#                              wotan batman-package emcee corner lightgbm \
#                              scikit-learn imbalanced-learn astroquery pyarrow
import sys, os
for p in [".", "..", r"g:\Exoplanet project"]:
    if os.path.isdir(os.path.join(p, "exopipeline")):
        sys.path.insert(0, os.path.abspath(p)); break
import numpy as np, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
from exopipeline import ingest, detrend, search, injection, config
print("exopipeline ready")


## 1. Load a quiet star and flatten it
We inject into the detrended flux so the injected transit is what we try to recover.

In [ ]:
star = ingest.clean(ingest.load_star("TIC 307210830", max_sectors=4))
flat = detrend.detrend(star.time, star.flux)
# Mask any real transits so the baseline is clean for injection
base = flat.flux.copy()
print(f"baseline: {len(flat.time)} points")

## 2. Run the injection-recovery grid
Reduce `n_inject` for a quick run; increase for a smoother curve.

In [ ]:
result = injection.run_injection_recovery(
    flat.time, base, n_inject=80,
    period_range=(2.0, 20.0), rp_range=(0.008, 0.04))
print(f"~50% completeness at SNR ≈ {result.snr50:.1f}")

## 3. Completeness curve

In [ ]:
fig = injection.plot_recovery(result, save_path="injection_recovery.png")
plt.show()
print("Saved injection_recovery.png")